# 06b - Modélisation ML : Prédiction blocs électoraux 2025–2027 — France métropolitaine

**Objectif** : Prédire le `bloc_dominant` (Gauche / Centre / Droite / ExtremeDroite) par commune pour 2025, 2026 et 2027 sur ~34 800 communes.

**Données** : `gold_france.features_communes` — ~174 000 lignes, ~34 800 communes × 5 années (2002, 2007, 2012, 2017, 2022)

**Approche** :
- Split temporel : train ≤ 2017, test = 2022
- Modèles : Random Forest + GradientBoosting
- Features : CSP, diplômes, démographie, tendances temporelles (lag, évolution), **typologie_territoire**
- Prédictions : extrapolation tendancielle 2025, 2026, 2027
- Résultats → `gold_france.predictions_2025_2027`

## 0. Configuration & imports

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Configuration inline
PG_HOST     = os.environ.get('POSTGRES_HOST',     'postgres')
PG_PORT     = os.environ.get('POSTGRES_PORT',     '5432')
PG_DB       = os.environ.get('POSTGRES_DB',       'mspr813')
PG_USER     = os.environ.get('POSTGRES_USER',     'mspr_user')
PG_PASSWORD = os.environ.get('POSTGRES_PASSWORD', 'mspr_password')

DB_URL = f'postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}'
engine = create_engine(DB_URL, pool_pre_ping=True)

print('PostgreSQL OK :', engine.connect().execute(text('SELECT current_database()')).scalar())

PostgreSQL OK : mspr813


## 1. Chargement des données Gold France

In [2]:
print('Chargement gold_france.features_communes...')
df = pd.read_sql(
    'SELECT * FROM gold_france.features_communes ORDER BY code_commune, annee',
    engine
)

print(f'Shape : {df.shape}')
print(f'Communes : {df["code_commune"].nunique()}')
print(f'Années   : {sorted(df["annee"].unique())}')
print(f'Blocs    : {df["bloc_dominant"].value_counts().to_dict()}')
print(f'\nTypologies :')
print(df['typologie_territoire'].value_counts().to_string())
df.head(3)

Chargement gold_france.features_communes...


Shape : (167866, 24)
Communes : 34800
Années   : [2002, 2007, 2012, 2017, 2022]
Blocs    : {'Droite': 64646, 'Gauche': 53037, 'ExtremeDroite': 36105, 'Centre': 14078}

Typologies :
typologie_territoire
rural         142836
periurbain     20633
urbain          4397


,code_commune,libelle,code_dep,annee,pct_gauche,pct_centre,pct_droite,pct_extremedroite,pct_divers,bloc_dominant,...,pct_bac_plus,pct_sans_diplome,nb_naissances,nb_deces,solde_naturel,typologie_territoire,created_at,mediane_revenu_disp,gini,taux_pauvrete
0,01001,ABERGEMENT CLEMENCIAT,01,2002,28.806,14.286,32.319,24.590,0.0,Droite,...,29.601,15.491,8,4,4,rural,2026-03-04 15:56:47.350049,25820.0,0.242,6.0
1,01001,ABERGEMENT CLEMENCIAT,01,2007,24.904,19.732,43.487,11.877,0.0,Droite,...,29.601,15.491,8,4,4,rural,2026-03-04 15:56:47.350049,25820.0,0.242,6.0
2,01001,ABERGEMENT CLEMENCIAT,01,2012,30.862,10.822,33.066,25.251,0.0,Droite,...,29.601,15.491,5,7,-2,rural,2026-03-04 15:56:47.350049,25820.0,0.242,6.0


## 2. Feature Engineering temporel

Ajout de features dérivées : tendances entre élections, lag, encodage de la typologie.

In [3]:
df = df.sort_values(['code_commune', 'annee']).reset_index(drop=True)

# Lag : résultat de l'élection précédente (par commune)
for col in ['pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite']:
    df[f'{col}_lag1'] = df.groupby('code_commune')[col].shift(1)

# Tendance : évolution par rapport à t-1
for col in ['pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite']:
    df[f'{col}_trend'] = df[col] - df[f'{col}_lag1']

# Bloc dominant à t-1
df['bloc_lag1'] = df.groupby('code_commune')['bloc_dominant'].shift(1)

# Intervalle depuis la dernière élection (années)
df['annee_lag1'] = df.groupby('code_commune')['annee'].shift(1)
df['intervalle'] = df['annee'] - df['annee_lag1']

# Encodage ordinal de la typologie : rural=0, périurbain=1, urbain=2
typo_map = {'rural': 0, 'periurbain': 1, 'urbain': 2}
df['typo_ord'] = df['typologie_territoire'].map(typo_map).fillna(0).astype(int)

print(f'Shape après features temporelles : {df.shape}')
df[['code_commune','annee','pct_gauche','pct_gauche_lag1','pct_gauche_trend','bloc_lag1','typo_ord']].head(10)

Shape après features temporelles : (167866, 36)


,code_commune,annee,pct_gauche,pct_gauche_lag1,pct_gauche_trend,bloc_lag1,typo_ord
0,01001,2002,28.806,NaN,NaN,NaN,0
1,01001,2007,24.904,28.806,-3.902,Droite,0
2,01001,2012,30.862,24.904,5.958,Droite,0
3,01001,2017,19.394,30.862,-11.468,Droite,0
4,01001,2022,21.731,19.394,2.337,Droite,0
5,01002,2002,31.293,NaN,NaN,NaN,0
6,01002,2007,29.825,31.293,-1.468,ExtremeDroite,0
7,01002,2012,37.931,29.825,8.106,Droite,0
8,01002,2017,28.409,37.931,-9.522,Gauche,0
9,01002,2022,38.596,28.409,10.187,Gauche,0


In [4]:
# Définir les features et la target
FEATURES = [
    # Résultats précédents (lag)
    'pct_gauche_lag1', 'pct_centre_lag1', 'pct_droite_lag1', 'pct_extremedroite_lag1',
    # Tendances
    'pct_gauche_trend', 'pct_centre_trend', 'pct_droite_trend', 'pct_extremedroite_trend',
    # CSP
    'cadres_pct', 'ouvriers_pct', 'employes_pct', 'artisans_pct',
    # Diplômes
    'pct_bac_plus', 'pct_sans_diplome',
    # Démographie
    'nb_naissances', 'nb_deces', 'solde_naturel',
    # Territoire
    'typo_ord',
    # Revenus
    'mediane_revenu_disp', 'gini', 'taux_pauvrete',
    # Temporel
    'annee',
]

TARGET = 'bloc_dominant'

# Supprimer les lignes sans lag (première élection par commune)
df_ml = df[df['bloc_lag1'].notna()].copy()
print(f'Lignes avec lag disponible : {len(df_ml)} (sur {len(df)} total)')
print(f'Années dans df_ml : {sorted(df_ml["annee"].unique())}')

Lignes avec lag disponible : 133066 (sur 167866 total)
Années dans df_ml : [2007, 2012, 2017, 2022]


## 3. Split temporel : train ≤ 2017 / test = 2022

In [5]:
# Split : train sur 2002/2007/2012, test sur 2017
# La prediction cible sera 2022 (comparaison avec les resultats reels 2022)
df_train = df_ml[df_ml['annee'] <= 2012].copy()
df_test  = df_ml[df_ml['annee'] == 2017].copy()

print(f'Train : {len(df_train)} lignes | annees : {sorted(df_train["annee"].unique())}')
print(f'Test  : {len(df_test)}  lignes | annees : {sorted(df_test["annee"].unique())}')
print(f'\nDistribution target train :')
print(df_train[TARGET].value_counts().to_string())
print(f'\nDistribution target test :')
print(df_test[TARGET].value_counts().to_string())

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET]


Train : 63496 lignes | annees : [2007, 2012]
Test  : 34787  lignes | annees : [2017]

Distribution target train :
bloc_dominant
Gauche           32091
Droite           29489
ExtremeDroite     1404
Centre             512

Distribution target test :
bloc_dominant
ExtremeDroite    11815
Droite           10424
Gauche            8530
Centre            4018


## 4. Modèle Random Forest

In [6]:
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print('Entrainement Random Forest...')
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest — Accuracy (test 2017) : {acc_rf:.3f}')
print()
print(classification_report(y_test, y_pred_rf, zero_division=0))


Entrainement Random Forest...


Random Forest — Accuracy (test 2017) : 0.547



               precision    recall  f1-score   support

       Centre       0.74      0.08      0.14      4018
       Droite       0.94      0.38      0.54     10424
ExtremeDroite       0.79      0.54      0.64     11815
       Gauche       0.38      0.98      0.55      8530

     accuracy                           0.55     34787
    macro avg       0.71      0.50      0.47     34787
 weighted avg       0.73      0.55      0.53     34787



In [7]:
# Importance des features
feat_imp = pd.Series(
    rf_pipeline.named_steps['model'].feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print('Top 10 features importances (Random Forest):')
print(feat_imp.head(10).round(4).to_string())

Top 10 features importances (Random Forest):
pct_centre_trend           0.2088
pct_gauche_lag1            0.1402
pct_extremedroite_trend    0.1401
pct_extremedroite_lag1     0.1003
annee                      0.0919
pct_droite_trend           0.0756
pct_centre_lag1            0.0607
pct_gauche_trend           0.0506
pct_droite_lag1            0.0473
gini                       0.0178


## 5. Modèle Gradient Boosting

In [8]:
# Encoder les labels pour GBM
all_blocs = ['Centre', 'Divers', 'Droite', 'ExtremeDroite', 'Gauche']
le = LabelEncoder()
le.fit(all_blocs)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

print('Entrainement GradientBoosting...')
gbm = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
gbm.fit(X_train_imp, y_train_enc)
y_pred_gbm_enc = gbm.predict(X_test_imp)
y_pred_gbm = le.inverse_transform(y_pred_gbm_enc)

acc_gbm = accuracy_score(y_test, y_pred_gbm)
print(f'GradientBoosting — Accuracy (test 2017) : {acc_gbm:.3f}')
print()
print(classification_report(y_test, y_pred_gbm, zero_division=0))


Entrainement GradientBoosting...


GradientBoosting — Accuracy (test 2017) : 0.808



               precision    recall  f1-score   support

       Centre       0.94      0.36      0.52      4018
       Droite       0.85      0.90      0.87     10424
ExtremeDroite       0.99      0.75      0.86     11815
       Gauche       0.64      0.99      0.77      8530

     accuracy                           0.81     34787
    macro avg       0.85      0.75      0.76     34787
 weighted avg       0.85      0.81      0.80     34787



In [9]:
# Choisir le meilleur modèle
best_model_name = 'RandomForest' if acc_rf >= acc_gbm else 'GradientBoosting'
best_pipeline   = rf_pipeline if acc_rf >= acc_gbm else None
best_imputer    = imputer
best_model      = rf_pipeline.named_steps['model'] if acc_rf >= acc_gbm else gbm

print(f'Meilleur modèle : {best_model_name} (accuracy={max(acc_rf, acc_gbm):.3f})')

Meilleur modèle : GradientBoosting (accuracy=0.808)


## 6. Prédictions 2025, 2026, 2027

**Stratégie** : Pour chaque commune, on extrapole les tendances observées entre 2017 et 2022.
- `pct_X_lag1` = valeur 2022
- `pct_X_trend` = tendance linéaire entre 2017→2022
- Features structurelles (CSP, diplômes, typologie) = constantes (2022)
- Démographie = extrapolation linéaire


In [10]:
# Construire les features pour predire 2022
# Lag = resultats 2017 (derniere election connue avant 2022)
# Tendance = evolution 2012 -> 2017
df_2017 = df[df['annee'] == 2017].copy()
df_2012 = df[df['annee'] == 2012].copy().set_index('code_commune')

print(f'Communes avec donnees 2017 : {len(df_2017)}')
print(f'Communes avec donnees 2012 : {len(df_2012)}')

predictions_rows = []

for _, row in df_2017.iterrows():
    comm = row['code_commune']

    # Tendance 2012 -> 2017
    def get_trend_2017(col):
        val_2017 = row.get(col, np.nan)
        if comm in df_2012.index:
            val_2012 = df_2012.loc[comm, col]
            if pd.notna(val_2017) and pd.notna(val_2012):
                return float(val_2017) - float(val_2012)
        return 0.0

    feat_row = {
        # Lag = resultats 2017
        'pct_gauche_lag1':         row.get('pct_gauche', np.nan),
        'pct_centre_lag1':         row.get('pct_centre', np.nan),
        'pct_droite_lag1':         row.get('pct_droite', np.nan),
        'pct_extremedroite_lag1':  row.get('pct_extremedroite', np.nan),
        # Tendance 2012 -> 2017
        'pct_gauche_trend':        get_trend_2017('pct_gauche'),
        'pct_centre_trend':        get_trend_2017('pct_centre'),
        'pct_droite_trend':        get_trend_2017('pct_droite'),
        'pct_extremedroite_trend': get_trend_2017('pct_extremedroite'),
        # CSP
        'cadres_pct':              row.get('cadres_pct', np.nan),
        'ouvriers_pct':            row.get('ouvriers_pct', np.nan),
        'employes_pct':            row.get('employes_pct', np.nan),
        'artisans_pct':            row.get('artisans_pct', np.nan),
        # Diplomes
        'pct_bac_plus':            row.get('pct_bac_plus', np.nan),
        'pct_sans_diplome':        row.get('pct_sans_diplome', np.nan),
        # Demographie
        'nb_naissances':           row.get('nb_naissances', np.nan),
        'nb_deces':                row.get('nb_deces', np.nan),
        'solde_naturel':           row.get('solde_naturel', np.nan),
        # Territoire
        'typo_ord':                row.get('typo_ord', 0),
        # Revenus
        'mediane_revenu_disp':     row.get('mediane_revenu_disp', np.nan),
        'gini':                    row.get('gini', np.nan),
        'taux_pauvrete':           row.get('taux_pauvrete', np.nan),
        # Annee cible
        'annee':                   2022,
        # Metadata
        '_code_commune':           comm,
        '_libelle':                row.get('libelle', ''),
        '_code_dep':               row.get('code_dep', ''),
        '_typologie':              row.get('typologie_territoire', ''),
    }
    predictions_rows.append(feat_row)

df_pred_input = pd.DataFrame(predictions_rows)
print(f'Communes a predire pour 2022 : {len(df_pred_input)}')
df_pred_input.head(3)


Communes avec donnees 2017 : 34791
Communes avec donnees 2012 : 28735


Communes a predire pour 2022 : 34791


,pct_gauche_lag1,pct_centre_lag1,pct_droite_lag1,pct_extremedroite_lag1,pct_gauche_trend,pct_centre_trend,pct_droite_trend,pct_extremedroite_trend,cadres_pct,ouvriers_pct,...,solde_naturel,typo_ord,mediane_revenu_disp,gini,taux_pauvrete,annee,_code_commune,_libelle,_code_dep,_typologie
0,19.394,24.444,30.303,25.455,-11.468,13.622,-2.763,0.204,17.460,20.635,...,6,0,25820.0,0.242,6.0,2022,01001,ABERGEMENT CLEMENCIAT,01,rural
1,28.409,21.023,23.295,27.273,-9.522,8.379,-4.291,5.434,11.111,16.667,...,2,0,24480.0,0.242,10.0,2022,01002,ABERGEMENT DE VAREY,01,rural
2,29.247,21.575,23.264,25.837,-13.333,13.065,-4.349,4.539,15.630,25.821,...,190,2,21660.0,0.280,3.0,2022,01004,AMBERIEU EN BUGEY,01,urbain


In [11]:
# Predire 2022 avec le meilleur modele
X_pred = df_pred_input[FEATURES]

if best_model_name == 'RandomForest':
    y_pred_future      = rf_pipeline.predict(X_pred)
    y_pred_future_prob = rf_pipeline.predict_proba(X_pred)
    classes = rf_pipeline.classes_
else:
    X_pred_imp = best_imputer.transform(X_pred)
    y_pred_future_enc  = gbm.predict(X_pred_imp)
    y_pred_future      = le.inverse_transform(y_pred_future_enc)
    y_pred_future_prob = gbm.predict_proba(X_pred_imp)
    classes = le.classes_[gbm.classes_]

# Construire DataFrame resultats
df_results = df_pred_input[['_code_commune', '_libelle', '_code_dep', '_typologie']].copy()
df_results.columns = ['code_commune', 'libelle', 'code_dep', 'typologie_territoire']
df_results['annee'] = 2022
df_results['bloc_predit'] = y_pred_future

n_classes_proba = y_pred_future_prob.shape[1]
for i, cls in enumerate(classes):
    if i < n_classes_proba:
        df_results[f'prob_{cls.lower()}'] = y_pred_future_prob[:, i].round(3)

print(f'Predictions 2022 generees : {len(df_results)}')
print('\nDistribution predictions 2022 :')
print(df_results['bloc_predit'].value_counts().to_string())
df_results.head(6)


Predictions 2022 generees : 34791

Distribution predictions 2022 :
bloc_predit
Centre           15127
ExtremeDroite    10286
Droite            7164
Gauche            2214


,code_commune,libelle,code_dep,typologie_territoire,annee,bloc_predit,prob_centre,prob_droite,prob_extremedroite,prob_gauche
0,01001,ABERGEMENT CLEMENCIAT,01,rural,2022,Centre,0.998,0.002,0.000,0.000
1,01002,ABERGEMENT DE VAREY,01,rural,2022,ExtremeDroite,0.028,0.191,0.525,0.257
2,01004,AMBERIEU EN BUGEY,01,urbain,2022,Centre,0.984,0.004,0.007,0.005
3,01005,AMBERIEUX EN DOMBES,01,rural,2022,ExtremeDroite,0.004,0.001,0.994,0.001
4,01006,AMBLEON,01,rural,2022,Centre,0.506,0.232,0.018,0.245
5,01007,AMBRONAY,01,periurbain,2022,Centre,0.952,0.001,0.046,0.001


## 7. Sauvegarde des prédictions en base

In [12]:
# Ajouter colonne modele
df_results['modele'] = best_model_name

# Harmoniser les colonnes prob
for col in ['prob_gauche', 'prob_centre', 'prob_droite', 'prob_extremedroite']:
    if col not in df_results.columns:
        df_results[col] = np.nan

# Vider la table avant insertion (idempotent)
with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE gold_france.predictions_2022'))

insert_cols = [
    'code_commune', 'libelle', 'code_dep', 'annee', 'bloc_predit',
    'prob_gauche', 'prob_centre', 'prob_droite', 'prob_extremedroite',
    'modele', 'typologie_territoire'
]
df_results[insert_cols].to_sql(
    'predictions_2022',
    engine,
    schema='gold_france',
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)

with engine.connect() as conn:
    n = conn.execute(text('SELECT COUNT(*) FROM gold_france.predictions_2022')).scalar()
print(f'gold_france.predictions_2022 : {n} lignes inserees')


gold_france.predictions_2022 : 34791 lignes inserees


## 8. Résultats finaux

In [13]:
print('=' * 65)
print('RESULTATS MODELISATION FRANCE METROPOLITAINE')
print('=' * 65)
print(f'  Modele Random Forest    — accuracy test 2017 : {acc_rf:.3f}')
print(f'  Modele GradientBoosting — accuracy test 2017 : {acc_gbm:.3f}')
print(f'  Modele retenu : {best_model_name}')
print()
print(f'  Predictions 2022 : {len(df_results)} communes')
dist = df_results['bloc_predit'].value_counts()
for bloc, cnt in dist.items():
    print(f'       {bloc:20s} : {cnt:6d} communes ({cnt/len(df_results)*100:.1f}%)')
print('=' * 65)


RESULTATS MODELISATION FRANCE METROPOLITAINE
  Modele Random Forest    — accuracy test 2017 : 0.547
  Modele GradientBoosting — accuracy test 2017 : 0.808
  Modele retenu : GradientBoosting

  Predictions 2022 : 34791 communes
       Centre               :  15127 communes (43.5%)
       ExtremeDroite        :  10286 communes (29.6%)
       Droite               :   7164 communes (20.6%)
       Gauche               :   2214 communes (6.4%)


In [14]:
# Repartition predictions 2022 par typologie territoire
print('Predictions 2022 par typologie territoire :')
print(
    df_results.groupby(['typologie_territoire', 'bloc_predit'])
    .size()
    .unstack(fill_value=0)
    .to_string()
)


Predictions 2022 par typologie territoire :
bloc_predit           Centre  Droite  ExtremeDroite  Gauche
typologie_territoire                                       
periurbain              2483     681            929     216
rural                  12148    6352           9234    1796
urbain                   496     131            123     202


In [15]:
# Top 10 departements par nombre de communes ExtremeDroite predites en 2022
pred_xd = df_results[df_results['bloc_predit'] == 'ExtremeDroite']
if len(pred_xd) > 0:
    top_dep = pred_xd.groupby('code_dep').size().sort_values(ascending=False).head(10)
    print('Top 10 departements communes ExtremeDroite predites en 2022 :')
    print(top_dep.to_string())
else:
    print('Aucune commune ExtremeDroite predite en 2022')
    print('Distribution complete :')
    print(df_results['bloc_predit'].value_counts().to_string())


Top 10 departements communes ExtremeDroite predites en 2022 :
code_dep
02     587
62     520
60     461
80     426
59     411
57     377
27     349
54     322
51     313
67     281
